# Non-Negative Least Squares (NNLS) for Train Fitting

## A Practical Guide for Electrophysiology and Imaging Analysis

---

**Author:** Based on the iGluSnFR analysis pipeline  
**Target audience:** Researchers analyzing stimulus-evoked trains (calcium imaging, glutamate imaging, electrophysiology)

---

### What you'll learn:
1. **Why NNLS?** — The fundamental problem of overlapping events
2. **Mathematical foundations** — From ordinary least squares to constrained optimization
3. **Building the design matrix** — Kernels and temporal templates
4. **Practical implementation** — Step-by-step with real data
5. **Advanced topics** — Weighting, robustness, and template variants

---
## Part 1: The Problem — Why Can't We Just Measure Peaks?

### The overlapping events problem

When analyzing stimulus trains (e.g., 10 pulses at 20-50Hz), each response **has not fully decayed** before the next stimulus arrives.

This creates two major issues:
1. **Baseline accumulation**: Later events ride on top of earlier decays
2. **Amplitude contamination**: Measured peak ≠ true amplitude of that event

**Example:** At 50Hz (20ms ISI), if τ_decay = 15ms, the first event has only decayed to ~27% when the second stimulus arrives!

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import nnls

# Set up nice plotting defaults
plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['font.size'] = 11

print("Libraries loaded successfully!")

In [ ]:
# Demonstrate the overlap problem
def simple_kernel(t, tau_rise=0.002, tau_decay=0.015):
    """Simple difference-of-exponentials kernel (iGluSnFR-like response)."""
    t = np.maximum(t, 0)
    return (1 - np.exp(-t / tau_rise)) * np.exp(-t / tau_decay)

# Simulation parameters
dt = 0.0005  # 0.5ms sampling
t = np.arange(0, 0.5, dt)  # 500ms trace
n_pulses = 5
isi = 0.020  # 50Hz
stim_times = np.arange(n_pulses) * isi + 0.050  # Start at 50ms

# True amplitudes (simulating paired-pulse facilitation)
true_amplitudes = np.array([1.0, 1.3, 1.5, 1.6, 1.65])

# Build the true signal (sum of scaled kernels)
signal_clean = np.zeros_like(t)
individual_events = []
for i, (st, amp) in enumerate(zip(stim_times, true_amplitudes)):
    event = amp * simple_kernel(t - st)
    individual_events.append(event)
    signal_clean += event

# Add some noise
np.random.seed(42)
noise_level = 0.05
signal_noisy = signal_clean + np.random.randn(len(t)) * noise_level

# Visualize the problem
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Panel A: Individual events
ax = axes[0]
colors = plt.cm.viridis(np.linspace(0.2, 0.8, n_pulses))
for i, event in enumerate(individual_events):
    ax.plot(t * 1000, event, color=colors[i], label=f'Event {i+1} (A={true_amplitudes[i]:.2f})')
ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Time (ms)')
ax.set_ylabel('Amplitude (a.u.)')
ax.set_title('A) Individual Events (TRUE amplitudes)')
ax.legend(fontsize=9)
ax.set_xlim([40, 200])

# Panel B: Summed signal
ax = axes[1]
ax.plot(t * 1000, signal_noisy, 'k-', alpha=0.7, label='Noisy signal')
ax.plot(t * 1000, signal_clean, 'b-', linewidth=2, label='Clean signal')
for st in stim_times:
    ax.axvline(st * 1000, color='red', linestyle=':', alpha=0.5)
ax.set_xlabel('Time (ms)')
ax.set_ylabel('Measured signal')
ax.set_title('B) What We Actually Record (overlapped)')
ax.legend()
ax.set_xlim([40, 200])

# Panel C: The problem with peak detection
ax = axes[2]
measured_peaks = []
for i, st in enumerate(stim_times):
    # Find peak in window after stimulus
    window_mask = (t >= st) & (t < st + isi)
    if np.any(window_mask):
        peak_val = np.max(signal_clean[window_mask])
        measured_peaks.append(peak_val)

x_pos = np.arange(n_pulses) + 1
ax.bar(x_pos - 0.2, true_amplitudes, width=0.35, label='TRUE amplitudes', color='green', alpha=0.7)
ax.bar(x_pos + 0.2, measured_peaks, width=0.35, label='Measured peaks', color='red', alpha=0.7)
ax.set_xlabel('Pulse number')
ax.set_ylabel('Amplitude')
ax.set_title('C) Peak Detection OVERESTIMATES Later Events!')
ax.legend()
ax.set_xticks(x_pos)

plt.tight_layout()
plt.show()

# Quantify the error
print("\n=== Peak Detection Error ===")
for i in range(n_pulses):
    error_pct = (measured_peaks[i] - true_amplitudes[i]) / true_amplitudes[i] * 100
    print(f"Event {i+1}: True={true_amplitudes[i]:.2f}, Measured={measured_peaks[i]:.2f}, Error={error_pct:+.1f}%")

---
## Part 2: The Solution — Linear Decomposition with NNLS

### Key insight: The signal is a **linear sum** of known waveforms

If we know the **shape** of each event (the kernel/template), the recorded signal is:

$$y(t) = \sum_{i=1}^{N} A_i \cdot K(t - t_i) + \epsilon(t)$$

Where:
- $y(t)$ = recorded signal
- $A_i$ = amplitude of event $i$ (what we want!)
- $K(t - t_i)$ = kernel shifted to stimulus time $t_i$
- $\epsilon(t)$ = noise

### In matrix form:

$$\mathbf{y} = \mathbf{X} \cdot \mathbf{a} + \boldsymbol{\epsilon}$$

Where $\mathbf{X}$ is the **design matrix** with each column being a shifted kernel.

In [ ]:
def build_design_matrix(time, stim_times, tau_rise=0.002, tau_decay=0.015):
    """Build the design matrix X where each column is a shifted kernel."""
    n_timepoints = len(time)
    n_events = len(stim_times)
    
    X = np.zeros((n_timepoints, n_events))
    for i, st in enumerate(stim_times):
        X[:, i] = simple_kernel(time - st, tau_rise, tau_decay)
    
    return X

# Build the design matrix
X = build_design_matrix(t, stim_times)

# Visualize the design matrix
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Panel A: Design matrix columns
ax = axes[0]
for i in range(n_pulses):
    ax.plot(t * 1000, X[:, i], color=colors[i], linewidth=2, label=f'Column {i+1}')
ax.set_xlabel('Time (ms)')
ax.set_ylabel('Kernel value')
ax.set_title('Design Matrix Columns (each = shifted kernel)')
ax.legend()
ax.set_xlim([40, 200])

# Panel B: Design matrix as image
ax = axes[1]
im = ax.imshow(X.T, aspect='auto', cmap='viridis', 
               extent=[t[0]*1000, t[-1]*1000, n_pulses+0.5, 0.5])
ax.set_xlabel('Time (ms)')
ax.set_ylabel('Event number')
ax.set_title('Design Matrix X (events × time)')
plt.colorbar(im, ax=ax, label='Kernel value')

plt.tight_layout()
plt.show()

print(f"\nDesign matrix shape: {X.shape} (timepoints × events)")

### Why Not Just Use Ordinary Least Squares (OLS)?

The standard OLS solution is:

$$\mathbf{a}_{OLS} = (\mathbf{X}^T \mathbf{X})^{-1} \mathbf{X}^T \mathbf{y}$$

**Problem:** OLS can give **negative amplitudes**!

This is physically impossible for:
- Fluorescence (can't have negative photons)
- Synaptic currents in the expected direction
- Any response that must be ≥ 0

In [ ]:
# Demonstrate the OLS problem with a tricky case
# Simulate a case with depression (later events smaller)
true_amps_depression = np.array([1.0, 0.6, 0.3, 0.1, 0.02])  # Strong depression

signal_depression = np.zeros_like(t)
for st, amp in zip(stim_times, true_amps_depression):
    signal_depression += amp * simple_kernel(t - st)

# Add noise
np.random.seed(123)
signal_depression_noisy = signal_depression + np.random.randn(len(t)) * 0.03

# OLS solution
ols_solution = np.linalg.lstsq(X, signal_depression_noisy, rcond=None)[0]

# NNLS solution
nnls_solution, _ = nnls(X, signal_depression_noisy)

# Compare
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.plot(t * 1000, signal_depression_noisy, 'k-', alpha=0.5, label='Noisy data')
ax.plot(t * 1000, X @ ols_solution, 'r--', linewidth=2, label='OLS fit')
ax.plot(t * 1000, X @ nnls_solution, 'g-', linewidth=2, label='NNLS fit')
ax.set_xlabel('Time (ms)')
ax.set_ylabel('Signal')
ax.set_title('Fits to Depressing Train')
ax.legend()
ax.set_xlim([40, 200])

ax = axes[1]
x_pos = np.arange(n_pulses) + 1
width = 0.25
ax.bar(x_pos - width, true_amps_depression, width, label='TRUE', color='blue', alpha=0.7)
ax.bar(x_pos, ols_solution, width, label='OLS', color='red', alpha=0.7)
ax.bar(x_pos + width, nnls_solution, width, label='NNLS', color='green', alpha=0.7)
ax.axhline(0, color='gray', linestyle='--')
ax.set_xlabel('Pulse number')
ax.set_ylabel('Amplitude')
ax.set_title('Amplitude Estimates')
ax.legend()
ax.set_xticks(x_pos)

plt.tight_layout()
plt.show()

print("\n=== Amplitude Comparison ===")
print(f"{'Event':<8} {'TRUE':>8} {'OLS':>8} {'NNLS':>8}")
print("-" * 36)
for i in range(n_pulses):
    print(f"{i+1:<8} {true_amps_depression[i]:>8.3f} {ols_solution[i]:>8.3f} {nnls_solution[i]:>8.3f}")

if np.any(ols_solution < 0):
    print("\n⚠️  OLS produced NEGATIVE amplitudes! This is physically impossible.")
print("✓ NNLS guarantees all amplitudes ≥ 0")

---
## Part 3: How NNLS Works

### The NNLS Problem

$$\min_{\mathbf{a}} \|\mathbf{X}\mathbf{a} - \mathbf{y}\|_2^2 \quad \text{subject to} \quad \mathbf{a} \geq 0$$

This is a **constrained quadratic optimization problem**.

### Key properties:
1. **Convex problem** → guaranteed global minimum
2. **Unique solution** (if X has full column rank)
3. **Sparse solutions** common (some amplitudes = exactly 0)

### The Algorithm (Lawson-Hanson, 1974)

The classic algorithm works by:
1. Start with all variables at 0 (in the "passive" set)
2. Iteratively move variables to the "active" set (allowed to be positive)
3. Check if any active variable wants to go negative → move back to passive
4. Repeat until convergence

**Good news:** `scipy.optimize.nnls` handles all this for you!

In [ ]:
# Simple demonstration of NNLS vs OLS
from scipy.optimize import nnls

# Simple 2D example for visualization
np.random.seed(42)

# Design matrix (2 predictors)
X_simple = np.array([[1.0, 0.5],
                     [0.8, 0.7],
                     [0.3, 1.0],
                     [0.1, 0.9]])

# True coefficients (one negative to show the problem)
a_true = np.array([1.5, -0.3])  # Second coefficient is negative
y_simple = X_simple @ a_true + np.random.randn(4) * 0.1

# Solve both ways
a_ols = np.linalg.lstsq(X_simple, y_simple, rcond=None)[0]
a_nnls, residual = nnls(X_simple, y_simple)

print("=== NNLS vs OLS Comparison ===")
print(f"\nTrue coefficients:    [{a_true[0]:+.3f}, {a_true[1]:+.3f}]")
print(f"OLS solution:         [{a_ols[0]:+.3f}, {a_ols[1]:+.3f}]")
print(f"NNLS solution:        [{a_nnls[0]:+.3f}, {a_nnls[1]:+.3f}]")

# Compute residuals
res_ols = np.sum((X_simple @ a_ols - y_simple)**2)
res_nnls = np.sum((X_simple @ a_nnls - y_simple)**2)

print(f"\nResidual sum of squares:")
print(f"  OLS:  {res_ols:.4f}")
print(f"  NNLS: {res_nnls:.4f}")
print(f"\n→ NNLS has slightly higher residual (it's constrained!) but guarantees a ≥ 0")

---
## Part 4: The Kernel (Template) — Heart of the Method

The quality of NNLS decomposition depends critically on having a **good kernel**.

### Common kernel models:

| Model | Formula | Use case |
|-------|---------|----------|
| Single exponential | $e^{-t/\tau_d}$ | Simple decay (rarely used) |
| Difference of exponentials | $(1 - e^{-t/\tau_r}) \cdot e^{-t/\tau_d}$ | Most biosensors, EPSCs |
| Double exponential decay | $f \cdot e^{-t/\tau_{fast}} + (1-f) \cdot e^{-t/\tau_{slow}}$ | Calcium indicators |
| iGluSnFR (tri-exponential) | Rise × (fast + slow + superslow decay) | Glutamate imaging |

### Key parameters:
- **τ_rise**: Rise time constant (typically 0.5-3 ms for fast sensors)
- **τ_decay**: Decay time constant (varies widely: 5-100+ ms)

In [ ]:
def kernel_single_exp(t, tau_decay=0.015):
    """Single exponential decay."""
    t = np.maximum(t, 0)
    return np.exp(-t / tau_decay)

def kernel_diff_exp(t, tau_rise=0.002, tau_decay=0.015):
    """Difference of exponentials (rise and decay)."""
    t = np.maximum(t, 0)
    k = (1 - np.exp(-t / tau_rise)) * np.exp(-t / tau_decay)
    # Normalize to peak = 1
    k = k / np.max(k) if np.max(k) > 0 else k
    return k

def kernel_biexp_decay(t, tau_rise=0.002, tau_fast=0.005, tau_slow=0.020, frac_fast=0.7):
    """Bi-exponential decay (fast + slow components)."""
    t = np.maximum(t, 0)
    rise = (1 - np.exp(-t / tau_rise))
    decay = frac_fast * np.exp(-t / tau_fast) + (1 - frac_fast) * np.exp(-t / tau_slow)
    k = rise * decay
    k = k / np.max(k) if np.max(k) > 0 else k
    return k

# Visualize different kernels
t_kernel = np.arange(0, 0.100, 0.0001)  # 100ms at high resolution

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Panel A: Different kernel shapes
ax = axes[0]
ax.plot(t_kernel * 1000, kernel_single_exp(t_kernel, 0.015), 'b-', label='Single exp (τ=15ms)', linewidth=2)
ax.plot(t_kernel * 1000, kernel_diff_exp(t_kernel, 0.002, 0.015), 'g-', label='Diff exp (τr=2ms, τd=15ms)', linewidth=2)
ax.plot(t_kernel * 1000, kernel_biexp_decay(t_kernel, 0.002, 0.005, 0.025, 0.6), 'r-', 
        label='Bi-exp (τr=2, τf=5, τs=25ms)', linewidth=2)
ax.set_xlabel('Time (ms)')
ax.set_ylabel('Normalized amplitude')
ax.set_title('Common Kernel Shapes')
ax.legend()
ax.set_xlim([0, 80])

# Panel B: Effect of decay time constant
ax = axes[1]
tau_values = [0.005, 0.010, 0.020, 0.040]
for tau in tau_values:
    ax.plot(t_kernel * 1000, kernel_diff_exp(t_kernel, 0.002, tau), 
            label=f'τ_decay = {tau*1000:.0f}ms', linewidth=2)
ax.set_xlabel('Time (ms)')
ax.set_ylabel('Normalized amplitude')
ax.set_title('Effect of Decay Time Constant')
ax.legend()
ax.set_xlim([0, 80])

plt.tight_layout()
plt.show()

### Why τ_decay matters for train analysis

The ratio of **τ_decay / ISI** determines how much overlap occurs:

| τ_decay / ISI | Overlap at next stim | Decomposition difficulty |
|---------------|---------------------|-------------------------|
| < 0.5 | < 37% | Easy |
| 0.5 - 1.0 | 37-63% | Moderate |
| 1.0 - 2.0 | 63-86% | Challenging |
| > 2.0 | > 86% | Very difficult |

In [ ]:
# Demonstrate overlap at different frequencies
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

tau_decay = 0.015  # 15ms decay
frequencies = [10, 20, 50]  # Hz

for idx, freq in enumerate(frequencies):
    ax = axes[idx]
    isi = 1.0 / freq
    stim_times_demo = np.arange(5) * isi + 0.020
    t_demo = np.arange(0, stim_times_demo[-1] + 0.150, dt)
    
    # Build individual events
    total = np.zeros_like(t_demo)
    for i, st in enumerate(stim_times_demo):
        event = kernel_diff_exp(t_demo - st, 0.002, tau_decay)
        ax.plot(t_demo * 1000, event, '--', alpha=0.4, color=colors[i])
        total += event
    
    ax.plot(t_demo * 1000, total, 'k-', linewidth=2, label='Sum')
    for st in stim_times_demo:
        ax.axvline(st * 1000, color='red', linestyle=':', alpha=0.5)
    
    overlap = np.exp(-isi / tau_decay) * 100
    ax.set_xlabel('Time (ms)')
    ax.set_ylabel('Amplitude')
    ax.set_title(f'{freq}Hz (ISI={isi*1000:.0f}ms)\nOverlap: {overlap:.0f}%')

plt.tight_layout()
plt.show()

---
## Part 5: Complete NNLS Pipeline — Step by Step

Let's put it all together with a realistic example.

In [ ]:
def nnls_decompose_train(
    signal, 
    time, 
    stim_times, 
    tau_rise=0.002, 
    tau_decay=0.015,
    kernel_func=kernel_diff_exp
):
    """
    Decompose a stimulus train using NNLS.
    
    Parameters
    ----------
    signal : array
        The recorded signal (baseline-subtracted)
    time : array
        Time vector
    stim_times : array
        Times of each stimulus
    tau_rise, tau_decay : float
        Kernel time constants (seconds)
    
    Returns
    -------
    amplitudes : array
        Estimated amplitude for each event
    reconstruction : array
        Reconstructed signal from the fit
    residual : array
        Difference between signal and reconstruction
    """
    # Step 1: Build design matrix
    n_events = len(stim_times)
    X = np.zeros((len(time), n_events))
    for i, st in enumerate(stim_times):
        X[:, i] = kernel_func(time - st, tau_rise, tau_decay)
    
    # Step 2: Solve NNLS
    amplitudes, nnls_residual = nnls(X, signal)
    
    # Step 3: Reconstruct signal
    reconstruction = X @ amplitudes
    residual = signal - reconstruction
    
    return amplitudes, reconstruction, residual, X


# Create a realistic test case
np.random.seed(2024)

# Parameters
dt = 0.0005
t = np.arange(0, 0.6, dt)
n_pulses = 10
isi = 0.020  # 50Hz
stim_times = np.arange(n_pulses) * isi + 0.050

# True amplitudes with PPR pattern (facilitation then depression)
true_amps = np.array([1.0, 1.4, 1.6, 1.7, 1.65, 1.55, 1.45, 1.35, 1.25, 1.20])

tau_r_true = 0.002
tau_d_true = 0.012

# Generate clean signal
signal_true = np.zeros_like(t)
for st, amp in zip(stim_times, true_amps):
    signal_true += amp * kernel_diff_exp(t - st, tau_r_true, tau_d_true)

# Add realistic noise
noise_level = 0.08
signal_noisy = signal_true + np.random.randn(len(t)) * noise_level

# Run NNLS decomposition
amps_nnls, recon, resid, X = nnls_decompose_train(
    signal_noisy, t, stim_times, 
    tau_rise=tau_r_true, tau_decay=tau_d_true
)

# Compare with naive peak detection
peaks_naive = []
for i, st in enumerate(stim_times):
    if i < len(stim_times) - 1:
        window_end = stim_times[i + 1]
    else:
        window_end = st + isi
    mask = (t >= st) & (t < window_end)
    peaks_naive.append(np.max(signal_noisy[mask]))
peaks_naive = np.array(peaks_naive)

print("NNLS Decomposition Complete!")

In [ ]:
# Comprehensive visualization
fig = plt.figure(figsize=(15, 10))

# Panel A: Raw signal and fit
ax1 = fig.add_subplot(2, 2, 1)
ax1.plot(t * 1000, signal_noisy, 'gray', alpha=0.7, label='Noisy signal')
ax1.plot(t * 1000, signal_true, 'b--', linewidth=1.5, label='True signal')
ax1.plot(t * 1000, recon, 'r-', linewidth=2, label='NNLS reconstruction')
for st in stim_times:
    ax1.axvline(st * 1000, color='green', linestyle=':', alpha=0.3)
ax1.set_xlabel('Time (ms)')
ax1.set_ylabel('Signal (a.u.)')
ax1.set_title('A) Signal Decomposition')
ax1.legend(loc='upper right')
ax1.set_xlim([40, 350])

# Panel B: Residuals
ax2 = fig.add_subplot(2, 2, 2)
ax2.plot(t * 1000, resid, 'k-', alpha=0.7)
ax2.axhline(0, color='red', linestyle='--')
ax2.axhline(noise_level * 2, color='orange', linestyle=':', label='±2σ')
ax2.axhline(-noise_level * 2, color='orange', linestyle=':')
ax2.set_xlabel('Time (ms)')
ax2.set_ylabel('Residual')
ax2.set_title(f'B) Residuals (should look like noise)')
ax2.legend()
ax2.set_xlim([40, 350])

# Panel C: Amplitude comparison
ax3 = fig.add_subplot(2, 2, 3)
x_pos = np.arange(n_pulses) + 1
width = 0.25
ax3.bar(x_pos - width, true_amps, width, label='TRUE', color='blue', alpha=0.7)
ax3.bar(x_pos, amps_nnls, width, label='NNLS', color='green', alpha=0.7)
ax3.bar(x_pos + width, peaks_naive, width, label='Peak detection', color='red', alpha=0.7)
ax3.set_xlabel('Pulse number')
ax3.set_ylabel('Amplitude')
ax3.set_title('C) Amplitude Estimates')
ax3.legend()
ax3.set_xticks(x_pos)

# Panel D: PPR comparison
ax4 = fig.add_subplot(2, 2, 4)
ppr_true = true_amps / true_amps[0]
ppr_nnls = amps_nnls / amps_nnls[0]
ppr_naive = peaks_naive / peaks_naive[0]

ax4.plot(x_pos, ppr_true, 'bo-', linewidth=2, markersize=8, label='TRUE')
ax4.plot(x_pos, ppr_nnls, 'gs-', linewidth=2, markersize=8, label='NNLS')
ax4.plot(x_pos, ppr_naive, 'r^-', linewidth=2, markersize=8, label='Peak detection')
ax4.axhline(1, color='gray', linestyle='--', alpha=0.5)
ax4.set_xlabel('Pulse number')
ax4.set_ylabel('Paired-Pulse Ratio (Pn/P1)')
ax4.set_title('D) Short-Term Plasticity Estimates')
ax4.legend()
ax4.set_xticks(x_pos)

plt.tight_layout()
plt.show()

# Quantitative comparison
print("\n" + "="*60)
print("QUANTITATIVE COMPARISON")
print("="*60)

# Amplitude errors
err_nnls = np.mean(np.abs(amps_nnls - true_amps))
err_peak = np.mean(np.abs(peaks_naive - true_amps))

print(f"\nMean absolute amplitude error:")
print(f"  NNLS:          {err_nnls:.3f}")
print(f"  Peak detection: {err_peak:.3f}")
print(f"  → NNLS is {err_peak/err_nnls:.1f}x more accurate")

# PPR errors (more meaningful biologically)
err_ppr_nnls = np.mean(np.abs(ppr_nnls - ppr_true))
err_ppr_peak = np.mean(np.abs(ppr_naive - ppr_true))

print(f"\nMean absolute PPR error:")
print(f"  NNLS:          {err_ppr_nnls:.3f}")
print(f"  Peak detection: {err_ppr_peak:.3f}")
print(f"  → NNLS is {err_ppr_peak/err_ppr_nnls:.1f}x more accurate for plasticity!")

---
## Part 6: Advanced Topics

### 6.1 Weighted NNLS

Not all timepoints are equally informative:
- **Peak regions** contain most amplitude information
- **Baseline regions** are mostly noise
- **Tail regions** can bias estimates (asymmetric noise)

Weighted NNLS minimizes:

$$\min_{\mathbf{a} \geq 0} \sum_t w_t \cdot (y_t - [\mathbf{X}\mathbf{a}]_t)^2$$

In [ ]:
def nnls_weighted(X, y, weights):
    """
    Weighted NNLS: minimize sum(w * (y - Xa)^2) subject to a >= 0
    
    Trick: Reweight by sqrt(w) to convert to standard NNLS form.
    """
    sqrt_w = np.sqrt(weights)
    X_weighted = X * sqrt_w[:, np.newaxis]  # Scale each row
    y_weighted = y * sqrt_w
    return nnls(X_weighted, y_weighted)

# Demonstrate different weighting schemes
def compute_weights(t, stim_times, isi, mode='uniform'):
    """Compute NNLS weights based on different strategies."""
    weights = np.ones_like(t)
    
    if mode == 'uniform':
        return weights
    
    elif mode == 'peak_emphasis':
        # Higher weight in windows after each stimulus
        peak_window = 0.010  # 10ms
        for st in stim_times:
            mask = (t >= st) & (t < st + peak_window)
            weights[mask] = 3.0  # 3x weight in peak window
        return weights
    
    elif mode == 'signal_amplitude':
        # Weight proportional to expected signal amplitude
        # Approximate by building a rough template fit first
        template_sum = np.zeros_like(t)
        for st in stim_times:
            template_sum += kernel_diff_exp(t - st, 0.002, 0.015)
        template_sum = np.abs(template_sum)
        weights = 0.1 + 0.9 * (template_sum / np.max(template_sum))
        return weights
    
    elif mode == 'last_event_downweight':
        # Downweight the tail after last event (prevents overshoot)
        weights = np.ones_like(t)
        last_stim = stim_times[-1]
        tail_start = last_stim + 0.010  # After peak window
        tail_mask = t >= tail_start
        t_rel = t[tail_mask] - tail_start
        tau_downweight = isi  # Decay with ISI time constant
        weights[tail_mask] = np.exp(-t_rel / tau_downweight)
        return weights
    
    return weights

# Compare weighting schemes
weight_modes = ['uniform', 'peak_emphasis', 'signal_amplitude', 'last_event_downweight']

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

for idx, mode in enumerate(weight_modes):
    ax = axes.flat[idx]
    weights = compute_weights(t, stim_times, isi, mode)
    
    ax.fill_between(t * 1000, 0, weights, alpha=0.3, color='blue')
    ax.plot(t * 1000, weights, 'b-', linewidth=1.5)
    
    # Show signal for reference
    ax2 = ax.twinx()
    ax2.plot(t * 1000, signal_noisy, 'gray', alpha=0.5)
    ax2.set_ylabel('Signal', color='gray')
    
    for st in stim_times:
        ax.axvline(st * 1000, color='red', linestyle=':', alpha=0.3)
    
    ax.set_xlabel('Time (ms)')
    ax.set_ylabel('Weight', color='blue')
    ax.set_title(f"'{mode}'")
    ax.set_xlim([40, 350])

plt.tight_layout()
plt.show()

### 6.2 Robust NNLS (Handling Outliers)

Standard NNLS uses **squared error**, which is sensitive to outliers.

**Huber loss** provides robustness:

$$L_\delta(r) = \begin{cases} 
\frac{1}{2}r^2 & \text{if } |r| \leq \delta \\
\delta(|r| - \frac{1}{2}\delta) & \text{if } |r| > \delta
\end{cases}$$

**IRLS (Iteratively Reweighted Least Squares)** approximates this by:
1. Fit with current weights
2. Downweight points with large residuals
3. Repeat

In [ ]:
def nnls_robust(X, y, huber_delta=2.5, n_iter=5):
    """
    Robust NNLS using Huber loss with IRLS.
    
    Parameters
    ----------
    X : array (n_timepoints, n_events)
        Design matrix
    y : array (n_timepoints,)
        Signal to fit
    huber_delta : float
        Huber threshold (in units of residual std)
    n_iter : int
        Number of IRLS iterations
    
    Returns
    -------
    amplitudes : array
    """
    weights = np.ones_like(y)
    
    for iteration in range(n_iter):
        # Weighted NNLS
        sqrt_w = np.sqrt(weights)
        X_w = X * sqrt_w[:, np.newaxis]
        y_w = y * sqrt_w
        amplitudes, _ = nnls(X_w, y_w)
        
        # Compute residuals
        residuals = y - X @ amplitudes
        
        # Update weights using Huber weighting
        # Points with |r| > delta get downweighted
        abs_r = np.abs(residuals)
        sigma = np.median(abs_r) * 1.4826  # Robust std estimate
        threshold = huber_delta * sigma
        
        weights = np.where(
            abs_r <= threshold,
            1.0,
            threshold / (abs_r + 1e-10)
        )
    
    return amplitudes, weights

# Demonstrate with outliers
np.random.seed(99)
signal_with_outliers = signal_noisy.copy()

# Add some spikes (outliers)
n_outliers = 15
outlier_idx = np.random.choice(len(t), n_outliers, replace=False)
signal_with_outliers[outlier_idx] += np.random.randn(n_outliers) * 0.5

# Compare standard vs robust NNLS
amps_standard, _ = nnls(X, signal_with_outliers)
amps_robust, weights_robust = nnls_robust(X, signal_with_outliers, huber_delta=2.5)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Panel A: Signal with outliers
ax = axes[0]
ax.plot(t * 1000, signal_with_outliers, 'k-', alpha=0.7, label='Signal with outliers')
ax.scatter(t[outlier_idx] * 1000, signal_with_outliers[outlier_idx], 
           c='red', s=50, zorder=5, label='Outliers')
ax.set_xlabel('Time (ms)')
ax.set_ylabel('Signal')
ax.set_title('A) Signal with Outliers')
ax.legend()
ax.set_xlim([40, 350])

# Panel B: IRLS weights
ax = axes[1]
ax.plot(t * 1000, weights_robust, 'b-', linewidth=1)
ax.scatter(t[outlier_idx] * 1000, weights_robust[outlier_idx], 
           c='red', s=50, zorder=5)
ax.set_xlabel('Time (ms)')
ax.set_ylabel('IRLS Weight')
ax.set_title('B) Robust Weights (outliers downweighted)')
ax.set_xlim([40, 350])

# Panel C: Amplitude comparison
ax = axes[2]
x_pos = np.arange(n_pulses) + 1
width = 0.25
ax.bar(x_pos - width, true_amps, width, label='TRUE', color='blue', alpha=0.7)
ax.bar(x_pos, amps_standard, width, label='Standard NNLS', color='orange', alpha=0.7)
ax.bar(x_pos + width, amps_robust, width, label='Robust NNLS', color='green', alpha=0.7)
ax.set_xlabel('Pulse number')
ax.set_ylabel('Amplitude')
ax.set_title('C) Amplitude Estimates')
ax.legend()
ax.set_xticks(x_pos)

plt.tight_layout()
plt.show()

# Quantify
err_standard = np.mean(np.abs(amps_standard - true_amps))
err_robust = np.mean(np.abs(amps_robust - true_amps))
print(f"\nMean absolute error with outliers:")
print(f"  Standard NNLS: {err_standard:.3f}")
print(f"  Robust NNLS:   {err_robust:.3f}")

### 6.3 Per-Pulse Decay Evolution

In many systems, **τ_decay changes across the train** (slowing due to accumulation).

Advanced approaches:
- **Linear progression**: τ_decay(n) = τ_0 + slope × n
- **Free monotonic**: Constrained to increase across pulses
- **Template variants**: Multiple templates with different τ values

In [ ]:
# Demonstrate per-pulse tau evolution
tau_d_progression = np.linspace(0.010, 0.020, n_pulses)  # 10ms → 20ms

# Build signal with evolving tau
signal_evolving = np.zeros_like(t)
for i, (st, amp) in enumerate(zip(stim_times, true_amps)):
    signal_evolving += amp * kernel_diff_exp(t - st, tau_r_true, tau_d_progression[i])

signal_evolving_noisy = signal_evolving + np.random.randn(len(t)) * noise_level

# Compare: fixed tau vs matching tau
X_fixed = np.column_stack([kernel_diff_exp(t - st, tau_r_true, 0.015) for st in stim_times])
X_evolving = np.column_stack([kernel_diff_exp(t - st, tau_r_true, tau_d_progression[i]) 
                              for i, st in enumerate(stim_times)])

amps_fixed, _ = nnls(X_fixed, signal_evolving_noisy)
amps_evolving, _ = nnls(X_evolving, signal_evolving_noisy)

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

ax = axes[0]
ax.plot(np.arange(n_pulses) + 1, tau_d_progression * 1000, 'ko-', linewidth=2, markersize=8)
ax.set_xlabel('Pulse number')
ax.set_ylabel('τ_decay (ms)')
ax.set_title('A) True τ_decay Evolution')

ax = axes[1]
ax.plot(t * 1000, signal_evolving_noisy, 'gray', alpha=0.5)
ax.plot(t * 1000, X_fixed @ amps_fixed, 'r--', linewidth=2, label='Fixed τ fit')
ax.plot(t * 1000, X_evolving @ amps_evolving, 'g-', linewidth=2, label='Evolving τ fit')
ax.set_xlabel('Time (ms)')
ax.set_ylabel('Signal')
ax.set_title('B) Reconstruction Quality')
ax.legend()
ax.set_xlim([40, 350])

ax = axes[2]
x_pos = np.arange(n_pulses) + 1
width = 0.25
ax.bar(x_pos - width, true_amps, width, label='TRUE', color='blue', alpha=0.7)
ax.bar(x_pos, amps_fixed, width, label='Fixed τ', color='red', alpha=0.7)
ax.bar(x_pos + width, amps_evolving, width, label='Evolving τ', color='green', alpha=0.7)
ax.set_xlabel('Pulse number')
ax.set_ylabel('Amplitude')
ax.set_title('C) Amplitude Accuracy')
ax.legend()
ax.set_xticks(x_pos)

plt.tight_layout()
plt.show()

err_fixed = np.mean(np.abs(amps_fixed - true_amps))
err_evolving = np.mean(np.abs(amps_evolving - true_amps))
print(f"\nMean absolute error:")
print(f"  Fixed τ:    {err_fixed:.3f}")
print(f"  Evolving τ: {err_evolving:.3f}")
print(f"\n→ Matching τ evolution improves accuracy by {(err_fixed-err_evolving)/err_fixed*100:.1f}%")

---
## Part 7: Practical Tips and Common Pitfalls

### ✅ Do:
1. **Estimate kinetics first** — Fit τ_rise and τ_decay from averaged data or isolated events
2. **Baseline properly** — Subtract pre-stimulus baseline before NNLS
3. **Check residuals** — They should look like white noise
4. **Use weights** — Emphasize informative timepoints
5. **Validate with simulations** — Test your pipeline with known amplitudes

### ❌ Don't:
1. **Use wrong kinetics** — Mismatched τ values cause systematic errors
2. **Ignore trial averaging** — Single trials are noisy; average or use robust methods
3. **Trust amplitudes near zero** — Small NNLS amplitudes may be noise
4. **Forget about jitter** — Stimulus timing uncertainty can bias fits

### Quality metrics to report:
- **R²** of the reconstruction
- **Residual autocorrelation** (should be ~0)
- **Estimated τ values** used
- **Noise level** relative to signal

In [ ]:
def assess_fit_quality(signal, reconstruction):
    """Compute quality metrics for NNLS fit."""
    residual = signal - reconstruction
    
    # R-squared
    ss_res = np.sum(residual**2)
    ss_tot = np.sum((signal - np.mean(signal))**2)
    r_squared = 1 - (ss_res / ss_tot)
    
    # Residual autocorrelation (lag 1)
    r_centered = residual - np.mean(residual)
    autocorr = np.corrcoef(r_centered[:-1], r_centered[1:])[0, 1]
    
    # Residual std vs signal amplitude
    snr = np.max(np.abs(signal)) / np.std(residual)
    
    return {
        'r_squared': r_squared,
        'autocorr_lag1': autocorr,
        'snr': snr,
        'residual_std': np.std(residual)
    }

# Assess our fit
quality = assess_fit_quality(signal_noisy, recon)

print("\n" + "="*50)
print("FIT QUALITY METRICS")
print("="*50)
print(f"R²:                    {quality['r_squared']:.4f}")
print(f"Residual autocorr:     {quality['autocorr_lag1']:.4f} (ideal: ~0)")
print(f"Signal-to-noise:       {quality['snr']:.1f}")
print(f"Residual std:          {quality['residual_std']:.4f}")

if quality['r_squared'] > 0.95:
    print("\n✓ Excellent fit!")
elif quality['r_squared'] > 0.90:
    print("\n✓ Good fit")
else:
    print("\n⚠ Consider checking kinetics parameters")

if abs(quality['autocorr_lag1']) > 0.3:
    print("⚠ High residual autocorrelation - possible model mismatch")

---
## Summary

### Key takeaways:

1. **NNLS solves the overlap problem** by decomposing the signal into individual event contributions

2. **The constraint a ≥ 0 is essential** for physically meaningful amplitudes

3. **Good kinetics are critical** — estimate τ_rise and τ_decay carefully

4. **Weighting and robustness** improve estimates in realistic conditions

5. **Always validate** — use simulations, check residuals, report quality metrics

### Further reading:
- Lawson & Hanson (1974) - Original NNLS algorithm
- Richardson & Bhalla (2004) - NNLS for calcium imaging
- Pnevmatikakis et al. (2016) - Constrained NMF for neural activity

### Code resources:
- `scipy.optimize.nnls` - Basic NNLS
- `sklearn.linear_model.Lasso` - L1-regularized (sparse) fits
- This repository's `extract_metrics.py` - Full iGluSnFR analysis pipeline

In [ ]:
print("\n" + "="*60)
print("🎉 Congratulations! You've completed the NNLS lecture.")
print("="*60)
print("\nNow try applying these concepts to your own data!")